# SQL Fundamentals with DuckDB

Your first SQL queries — no prior SQL experience needed.

Every query follows this pattern:

```sql
SELECT  what_you_want
FROM    table_name
WHERE   condition       -- filter rows   (optional)
GROUP BY column         -- group rows    (optional)
ORDER BY column DESC    -- sort result   (optional)
LIMIT   n               -- cap rows      (optional)
```

We query the Tanzania household survey using DuckDB, a fast SQL engine that runs
inside the notebook and reads pandas DataFrames directly.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import duckdb

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.append(str(project_root))

from src.utilities.project_paths import RAW_DIR

DTA_DIR = RAW_DIR / 'tanzania' / 'TZNPS5_20_11_STATA'

COLS_HH = ['interview__id', 't0_region', 't0_district',
           'nps4_hhsize', 'nps4_nplots', 'int_result']

households = pd.read_stata(
    DTA_DIR / 'TZNPS5_20.dta',
    columns=COLS_HH,
    convert_categoricals=False,
)
roster = pd.read_stata(
    DTA_DIR / 't2_roster.dta',
    columns=['interview__id', 'Calc_Age'],
    convert_categoricals=False,
).dropna()

duckdb.register('households', households)
duckdb.register('roster', roster)

print(f'households : {len(households):,} rows')
print(f'roster     : {len(roster):,} rows')
print()
print('households columns:', list(households.columns))

---
# Part A — Looking at the Data

| Clause | What it does |
|---|---|
| `SELECT *` | Return all columns |
| `SELECT col1, col2` | Return specific columns |
| `LIMIT n` | Return only the first n rows |

## A1. How many rows are there?

`COUNT(*)` counts every row in the table.

In [ ]:
# TODO: count all rows in the households table, name the result n_rows
duckdb.sql("""
    SELECT COUNT(*) AS ...
    FROM households
""").to_df()

## A2. Peek at the data

Show the first 5 rows with all columns.

In [ ]:
# TODO: select all columns, limit to 5 rows
duckdb.sql("""
    SELECT ...
    FROM households
    LIMIT ...
""").to_df()

## A3. Select specific columns

Show only `interview__id`, `t0_region`, and `nps4_hhsize` for the first 10 rows.

In [ ]:
# TODO: select those three columns, limit to 10 rows
duckdb.sql("""
    SELECT ..., ..., ...
    FROM households
    LIMIT 10
""").to_df()

---
# Part B — Filtering with WHERE

`WHERE` keeps only rows that match a condition.

| Operator | Meaning | Example |
|---|---|---|
| `=` | equal | `int_result = 1` |
| `>` / `<` | greater / less than | `nps4_hhsize > 5` |
| `>=` / `<=` | greater or equal / less or equal | `nps4_hhsize >= 3` |
| `AND` | both conditions must be true | `int_result = 1 AND nps4_hhsize > 5` |
| `OR` | at least one must be true | `t0_region = 1 OR t0_region = 2` |

## B1. Completed interviews only

`int_result = 1` means the interview was completed.
Show all columns, limit to 10 rows.

In [ ]:
# TODO: add a WHERE clause to filter int_result = 1
duckdb.sql("""
    SELECT *
    FROM households
    WHERE ...
    LIMIT 10
""").to_df()

## B2. Large households

Show households with more than 6 members.
Select `interview__id`, `t0_region`, `nps4_hhsize`.

In [ ]:
# TODO: WHERE nps4_hhsize > 6
duckdb.sql("""
    SELECT interview__id, t0_region, nps4_hhsize
    FROM households
    WHERE ...
""").to_df()

## B3. Combine two conditions

Show large households (> 6 members) from completed interviews only.
Use `AND` to apply both filters at once.

In [ ]:
# TODO: WHERE int_result = 1 AND nps4_hhsize > 6
# SELECT interview__id, t0_region, nps4_hhsize, nps4_nplots
duckdb.sql("""
    SELECT ...
    FROM households
    WHERE ... AND ...
""").to_df()

---
# Part C — Counting and Summarising

Aggregate functions collapse many rows into a single number.

| Function | Result |
|---|---|
| `COUNT(*)` | number of rows |
| `AVG(col)` | mean |
| `SUM(col)` | total |
| `MIN(col)` | smallest value |
| `MAX(col)` | largest value |

You can use several aggregates in a single `SELECT`.

## C1. How many completed interviews?

Count only the rows where `int_result = 1`.

In [ ]:
# TODO: WHERE int_result = 1, COUNT(*) AS n_complete
duckdb.sql("""
    SELECT COUNT(*) AS n_complete
    FROM households
    WHERE ...
""").to_df()

## C2. Summarise household size

Compute the average, minimum, and maximum of `nps4_hhsize` across all rows.

In [ ]:
# TODO: SELECT AVG(...) AS mean_size, MIN(...) AS min_size, MAX(...) AS max_size
duckdb.sql("""
    SELECT
        AVG(nps4_hhsize) AS mean_size,
        ...
    FROM households
""").to_df()

## C3. Multiple aggregations on filtered data

For completed interviews: count households, mean household size, total plots.

In [ ]:
# TODO: WHERE int_result = 1
# SELECT COUNT(*) AS n_hh, AVG(nps4_hhsize) AS mean_size, SUM(nps4_nplots) AS total_plots
duckdb.sql("""
    SELECT
        COUNT(*) AS n_hh,
        ...
    FROM households
    WHERE ...
""").to_df()

---
# Part D — Grouping with GROUP BY

`GROUP BY` splits rows into groups and applies aggregate functions to each group.

```sql
SELECT  group_column, AGG(value_column) AS alias
FROM    table
WHERE   condition
GROUP BY group_column
```

**Rule:** every column in `SELECT` must be either in `GROUP BY` or wrapped in an aggregate.

## D1. Count households per interview result

How many households have each value of `int_result`?

In [ ]:
# TODO: GROUP BY int_result, COUNT(*) AS n_hh
duckdb.sql("""
    SELECT
        int_result,
        COUNT(*) AS n_hh
    FROM households
    GROUP BY ...
""").to_df()

## D2. Average household size by region

For completed interviews, compute the number of households and mean household size in each region.

In [ ]:
# TODO:
# SELECT t0_region, COUNT(*) AS n_hh, AVG(nps4_hhsize) AS mean_size
# WHERE int_result = 1
# GROUP BY t0_region
duckdb.sql("""
    SELECT
        t0_region,
        ...
    FROM households
    WHERE ...
    GROUP BY ...
""").to_df()

---
# Part E — Sorting with ORDER BY

`ORDER BY` sorts the result rows.

```sql
ORDER BY column_name        -- ascending (smallest first, this is the default)
ORDER BY column_name DESC   -- descending (largest first)
```

Combine with `LIMIT` to get the top or bottom N rows.

## E1. Regions with the largest average household size

Repeat D2, but sort by `mean_size` descending so the largest regions appear first.

In [ ]:
# TODO: add ORDER BY mean_size DESC to the D2 query
duckdb.sql("""
    SELECT
        t0_region,
        COUNT(*)         AS n_hh,
        AVG(nps4_hhsize) AS mean_size
    FROM households
    WHERE int_result = 1
    GROUP BY t0_region
    ORDER BY ...
""").to_df()

## E2. The 5 largest individual households

Find the 5 households with the most members.
Show `interview__id`, `t0_region`, `nps4_hhsize`.

In [ ]:
# TODO: ORDER BY nps4_hhsize DESC, LIMIT 5
duckdb.sql("""
    SELECT interview__id, t0_region, nps4_hhsize
    FROM households
    WHERE int_result = 1
    ORDER BY ...
    LIMIT ...
""").to_df()

## E3. Regions with the fewest completed interviews

Sort ascending (`ASC`, the default) to find the smallest regions first.

In [ ]:
# TODO: GROUP BY t0_region, COUNT(*) AS n_hh, ORDER BY n_hh ASC
duckdb.sql("""
    SELECT
        t0_region,
        COUNT(*) AS n_hh
    FROM households
    WHERE int_result = 1
    GROUP BY t0_region
    ORDER BY ...
""").to_df()